In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
# ============================================================
#  🦴 Bone Fracture 10-Class — Multi-Model Comparison v4
#  Partial Fine-Tuning (마지막 블록 unfreeze + classifier)
# ============================================================
!pip install -q timm

import os, pathlib, time, json as _json, shutil, warnings, numpy as np
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler, Subset
from torchvision import transforms
from torchvision.datasets import ImageFolder
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import timm
from datetime import datetime
from IPython.display import display, HTML
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100

# ── 경로 설정 ────────────────────────────────────────────────
INPUT_PATH  = "/content/drive/MyDrive/2026_lecture/Medical_AI/Medical_Imagining/Bone Break Classification"
LOCAL_DATA  = "/content/bone_data"
OUTPUT_BASE = "/content/drive/MyDrive/2026_lecture/Medical_AI/1week/output"
RUN_STAMP   = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_PATH = os.path.join(OUTPUT_BASE, f"compare_{RUN_STAMP}")
os.makedirs(OUTPUT_PATH, exist_ok=True)

# ── 데이터 로컬 복사 ────────────────────────────────────────
if not os.path.exists(LOCAL_DATA):
    print("📂 Drive → Local SSD 복사 중...")
    shutil.copytree(INPUT_PATH, LOCAL_DATA)
    print("✅ 복사 완료!")
else:
    print("✅ 로컬 데이터 이미 존재")

# ── 하이퍼파라미터 ───────────────────────────────────────────
IMG_SIZE, BATCH_SIZE, NUM_EPOCHS, LR, SEED = 224, 32, 10, 3e-4, 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED); np.random.seed(SEED)
print(f"⚡ Device: {DEVICE}")

IMG_EXTS = {'.png','.jpg','.jpeg','.bmp','.tiff','.tif','.webp'}

class MultiExtImageFolder(ImageFolder):
    def is_valid_file(self, path):
        return pathlib.Path(path).suffix.lower() in IMG_EXTS

# ── 클래스 분석 ──────────────────────────────────────────────
class_names = sorted([d for d in os.listdir(LOCAL_DATA)
                      if os.path.isdir(os.path.join(LOCAL_DATA, d))])
NUM_CLASSES = len(class_names)
print(f"📂 Classes({NUM_CLASSES}): {class_names}")
class_counts = {}
for cls in class_names:
    count = len([f for f in pathlib.Path(os.path.join(LOCAL_DATA, cls)).rglob("*")
                 if f.suffix.lower() in IMG_EXTS])
    class_counts[cls] = count
total_images = sum(class_counts.values())

# ── ① 클래스 분포 차트 ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(range(NUM_CLASSES), [class_counts[c] for c in class_names],
              color=plt.cm.Set3.colors[:NUM_CLASSES], edgecolor='white')
ax.set_xticks(range(NUM_CLASSES))
ax.set_xticklabels(class_names, rotation=30, ha='right', fontsize=9)
ax.set_ylabel("Count")
ax.set_title(f"📊 Class Distribution (Total: {total_images})", fontweight='bold', fontsize=14)
for b, c in zip(bars, class_names):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+1,
            str(class_counts[c]), ha='center', fontsize=8, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_PATH, '01_class_distribution.png'), dpi=150)
plt.show()
print()

# ── Transforms ───────────────────────────────────────────────
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ── 데이터 분할 (80/10/10) ───────────────────────────────────
full_ds = MultiExtImageFolder(root=LOCAL_DATA)
n = len(full_ds)
indices = list(range(n)); np.random.shuffle(indices)
n_train = int(n * 0.8); n_val = int(n * 0.1)
train_idx = indices[:n_train]
val_idx   = indices[n_train:n_train+n_val]
test_idx  = indices[n_train+n_val:]

train_ds = Subset(MultiExtImageFolder(root=LOCAL_DATA, transform=train_tf), train_idx)
val_ds   = Subset(MultiExtImageFolder(root=LOCAL_DATA, transform=eval_tf), val_idx)
test_ds  = Subset(MultiExtImageFolder(root=LOCAL_DATA, transform=eval_tf), test_idx)
print(f"📦 Train:{len(train_ds)} | Val:{len(val_ds)} | Test:{len(test_ds)}")

# WeightedRandomSampler
labels_train = [full_ds.targets[i] for i in train_idx]
cls_count_arr = np.array([labels_train.count(i) for i in range(NUM_CLASSES)])
w = 1.0 / np.maximum(cls_count_arr, 1)
sample_w = np.array([w[l] for l in labels_train])
sampler = WeightedRandomSampler(torch.from_numpy(sample_w).float(),
                                 len(sample_w), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# ==============================================================
#  Partial Fine-Tuning: backbone 마지막 블록 + classifier 학습
# ==============================================================
def build_model_partial_ft(name):
    """
    전략: 전체 freeze → 마지막 블록 + classifier unfreeze
    - Backbone의 하위 레이어(범용 특징)는 보존
    - 상위 레이어(도메인 특화)만 재학습
    - Classifier는 Dropout + Linear로 교체
    """
    model = timm.create_model(name, pretrained=True, num_classes=NUM_CLASSES)

    # Step 1: 전체 freeze
    for param in model.parameters():
        param.requires_grad = False

    # Step 2: 마지막 블록 unfreeze (모델별 처리)
    if 'efficientnet' in name:
        for param in model.blocks[-1].parameters():  # 마지막 블록
            param.requires_grad = True
        for param in model.conv_head.parameters():
            param.requires_grad = True
        for param in model.bn2.parameters():
            param.requires_grad = True
    elif 'resnet' in name:
        for param in model.layer4.parameters():
            param.requires_grad = True
    elif 'densenet' in name:
        for param in model.features.denseblock4.parameters():
            param.requires_grad = True
        for param in model.features.norm5.parameters():
            param.requires_grad = True
    elif 'convnext' in name:
        for param in model.stages[-1].parameters():
            param.requires_grad = True
        for param in model.head.parameters():
            param.requires_grad = True
    elif 'mobilenet' in name:
        for param in model.blocks[-1].parameters():
            param.requires_grad = True
        for param in model.conv_head.parameters():
            param.requires_grad = True
        for param in model.bn2.parameters():
            param.requires_grad = True

    # Step 3: Classifier head 교체 (Dropout 추가)
    in_features = model.get_classifier().in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, NUM_CLASSES)
    )
    model = model.to(DEVICE)

    total_p = sum(p.numel() for p in model.parameters())
    train_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   Total: {total_p:,} | Trainable: {train_p:,} ({train_p/total_p*100:.1f}%)")
    return model

MODEL_LIST = [
    ("EfficientNet-B0", "efficientnet_b0"),
    ("ResNet-50",       "resnet50"),
    ("DenseNet-121",    "densenet121"),
    ("ConvNeXt-Tiny",   "convnext_tiny"),
    ("MobileNetV2",     "mobilenetv2_100"),
]

# ==============================================================
#  학습 + 평가
# ==============================================================
results = []

for model_label, model_name in MODEL_LIST:
    print(f"\n{'='*60}")
    print(f"🧠 {model_label} — Partial Fine-Tuning")
    print(f"{'='*60}")

    torch.manual_seed(SEED); np.random.seed(SEED)
    model = build_model_partial_ft(model_name)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                            lr=LR, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0.0
    t0 = time.time()

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train(); t_loss, t_ok, t_n = 0.0, 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            out = model(imgs); loss = criterion(out, labels)
            loss.backward(); optimizer.step()
            t_loss += loss.item()*imgs.size(0)
            t_ok += (out.argmax(1)==labels).sum().item(); t_n += imgs.size(0)

        model.eval(); v_loss, v_ok, v_n = 0.0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                out = model(imgs); loss = criterion(out, labels)
                v_loss += loss.item()*imgs.size(0)
                v_ok += (out.argmax(1)==labels).sum().item(); v_n += imgs.size(0)
        scheduler.step()

        tl, ta = t_loss/t_n, t_ok/t_n*100; vl, va = v_loss/v_n, v_ok/v_n*100
        history['train_loss'].append(tl); history['train_acc'].append(ta)
        history['val_loss'].append(vl); history['val_acc'].append(va)
        flag = ""
        if va > best_val_acc:
            best_val_acc = va
            torch.save(model.state_dict(), os.path.join(OUTPUT_PATH, f'best_{model_name}.pth'))
            flag = " ⭐"
        print(f"  Epoch {epoch:2d}/{NUM_EPOCHS} | "
              f"Train {tl:.3f}/{ta:.1f}% | Val {vl:.3f}/{va:.1f}%{flag}")

    elapsed = time.time() - t0

    # Test
    model.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f'best_{model_name}.pth')))
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in test_loader:
            preds = model(imgs.to(DEVICE)).argmax(1).cpu().numpy()
            all_preds.extend(preds); all_labels.extend(labels.numpy())
    all_preds = np.array(all_preds); all_labels = np.array(all_labels)
    test_acc = (all_preds == all_labels).mean() * 100
    cm = confusion_matrix(all_labels, all_preds)

    print(f"\n  ✅ {model_label} — Test Acc: {test_acc:.2f}% | Time: {elapsed:.1f}s")

    results.append({
        'label': model_label, 'name': model_name,
        'params_total': sum(p.numel() for p in model.parameters()),
        'params_trainable': sum(p.numel() for p in model.parameters() if p.requires_grad),
        'time': elapsed, 'best_val_acc': best_val_acc, 'test_acc': test_acc,
        'history': history, 'cm': cm, 'preds': all_preds, 'gt': all_labels,
    })
    del model; torch.cuda.empty_cache()

# ==============================================================
#  📊 시각화 — 모두 Colab 화면에 출력
# ==============================================================
short = [c.replace(" fracture","").replace(" Fracture","") for c in class_names]
colors5 = ['#4285F4','#EA4335','#FBBC04','#34A853','#FF6D01']
sorted_res = sorted(results, key=lambda x: -x['test_acc'])

# ── ② HTML 요약 테이블 ──────────────────────────────────────
html = "<h3>📊 종합 비교 — Partial Fine-Tuning (마지막 블록 + Classifier)</h3>"
html += "<table style='border-collapse:collapse; font-family:monospace; font-size:14px; margin:10px 0;'>"
html += ("<tr style='background:#4285F4; color:white;'>"
         "<th style='padding:8px 14px;'>Rank</th>"
         "<th style='padding:8px 14px;'>Model</th>"
         "<th style='padding:8px 14px;'>Total Params</th>"
         "<th style='padding:8px 14px;'>Trainable</th>"
         "<th style='padding:8px 14px;'>Train %</th>"
         "<th style='padding:8px 14px;'>Time</th>"
         "<th style='padding:8px 14px;'>Val Acc</th>"
         "<th style='padding:8px 14px;'>Test Acc</th></tr>")
for rank, r in enumerate(sorted_res, 1):
    bg = '#E8F5E9' if rank == 1 else ('#FFF' if rank % 2 == 0 else '#F5F5F5')
    medal = ['🥇','🥈','🥉'][rank-1] if rank <= 3 else f'  {rank}'
    pct = r['params_trainable']/r['params_total']*100
    html += (f"<tr style='background:{bg};'>"
             f"<td style='padding:6px 14px; text-align:center;'>{medal}</td>"
             f"<td style='padding:6px 14px; font-weight:bold;'>{r['label']}</td>"
             f"<td style='padding:6px 14px; text-align:right;'>{r['params_total']:,}</td>"
             f"<td style='padding:6px 14px; text-align:right;'>{r['params_trainable']:,}</td>"
             f"<td style='padding:6px 14px; text-align:right;'>{pct:.1f}%</td>"
             f"<td style='padding:6px 14px; text-align:right;'>{r['time']:.0f}s</td>"
             f"<td style='padding:6px 14px; text-align:right;'>{r['best_val_acc']:.1f}%</td>"
             f"<td style='padding:6px 14px; text-align:right; font-weight:bold; "
             f"color:{'#2E7D32' if rank==1 else '#333'};'>{r['test_acc']:.1f}%</td></tr>")
html += "</table>"
html += (f"<p style='font-size:12px; color:#888;'>IMG={IMG_SIZE} | BS={BATCH_SIZE} | "
         f"EP={NUM_EPOCHS} | LR={LR} | Data: {len(train_ds)}/{len(val_ds)}/{len(test_ds)}</p>")
display(HTML(html))

# ── ③ Accuracy / Time / Trainable Params ────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
lbl = [r['label'] for r in results]
accs = [r['test_acc'] for r in results]
tms  = [r['time'] for r in results]
trk  = [r['params_trainable']/1e3 for r in results]

for ax, vals, ylabel, title, fmt in zip(
    axes,
    [accs, tms, trk],
    ['Test Accuracy (%)', 'Training Time (sec)', 'Trainable Params (K)'],
    ['🏆 Test Accuracy', '⏱️ Training Time', '📐 Trainable Parameters'],
    ['{:.1f}%', '{:.0f}s', '{:.1f}K']
):
    bars = ax.bar(lbl, vals, color=colors5, edgecolor='white', linewidth=1.5)
    for b, v in zip(bars, vals):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+max(vals)*0.01,
                fmt.format(v), ha='center', fontsize=10, fontweight='bold')
    ax.set_ylabel(ylabel); ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylim(0, max(vals)*1.15)
    ax.tick_params(axis='x', rotation=20); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_PATH, '02_comparison_bars.png'), dpi=150)
plt.show(); print()

# ── ④ 학습 곡선 ─────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
for r, c in zip(results, colors5):
    ep = range(1, NUM_EPOCHS + 1)
    ax1.plot(ep, r['history']['val_loss'], '-o', color=c,
             label=r['label'], markersize=4, linewidth=2)
    ax2.plot(ep, r['history']['val_acc'], '-o', color=c,
             label=r['label'], markersize=4, linewidth=2)
ax1.set_title('Validation Loss', fontsize=13, fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.legend(fontsize=9); ax1.grid(alpha=0.3)
ax2.set_title('Validation Accuracy (%)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.legend(fontsize=9); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_PATH, '03_learning_curves.png'), dpi=150)
plt.show(); print()

# ── ⑤ Confusion Matrix (5개 나란히) ─────────────────────────
fig, axes_cm = plt.subplots(1, len(results), figsize=(6*len(results), 5.5))
if len(results) == 1: axes_cm = [axes_cm]
for ax, r in zip(axes_cm, results):
    sns.heatmap(r['cm'], annot=True, fmt='d', cmap='Blues',
                xticklabels=short, yticklabels=short, ax=ax,
                linewidths=0.5, cbar=False)
    ax.set_title(f"{r['label']}\nTest Acc: {r['test_acc']:.1f}%",
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.tick_params(axis='x', rotation=45, labelsize=6)
    ax.tick_params(axis='y', rotation=0, labelsize=6)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_PATH, '04_confusion_matrices.png'), dpi=120)
plt.show(); print()

# ── ⑥ Per-Class Accuracy 히트맵 ─────────────────────────────
pca_data = []
for r in results:
    rs = r['cm'].sum(axis=1)
    pca_data.append(np.where(rs > 0, r['cm'].diagonal()/rs*100, 0))
pca = np.array(pca_data)

fig, ax = plt.subplots(figsize=(14, 4.5))
sns.heatmap(pca, annot=True, fmt='.0f', cmap='RdYlGn',
            xticklabels=short, yticklabels=[r['label'] for r in results],
            ax=ax, linewidths=0.5, vmin=0, vmax=100,
            annot_kws={"fontsize": 9, "fontweight": "bold"})
ax.set_title('Per-Class Accuracy (%) — 어떤 골절을 잘/못 분류하는가?',
             fontsize=13, fontweight='bold')
ax.tick_params(axis='x', rotation=30, labelsize=9)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_PATH, '05_per_class_heatmap.png'), dpi=150)
plt.show(); print()

# ── ⑦ Efficiency 산점도 ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
for r, c in zip(results, colors5):
    ax.scatter(r['time'], r['test_acc'], s=350,
               c=c, alpha=0.85, edgecolors='black', linewidth=1.5, zorder=5)
    ax.annotate(r['label'], (r['time'], r['test_acc']),
                textcoords="offset points", xytext=(10, 8),
                fontsize=11, fontweight='bold')
ax.set_xlabel('Training Time (sec)', fontsize=12)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('⚡ Accuracy vs Speed — 우상단이 이상적', fontsize=13, fontweight='bold')
ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_PATH, '06_efficiency.png'), dpi=150)
plt.show(); print()

# ── ⑧ Best 모델 Classification Report ───────────────────────
best = max(results, key=lambda x: x['test_acc'])
print(f"{'='*60}")
print(f"🏆 Best: {best['label']} — Classification Report")
print(f"{'='*60}")
print(classification_report(best['gt'], best['preds'],
                            target_names=class_names, digits=3))

# ── JSON 저장 ────────────────────────────────────────────────
summary = {
    "run_timestamp": RUN_STAMP,
    "strategy": "Partial Fine-Tuning (last block + classifier)",
    "img_size": IMG_SIZE, "batch_size": BATCH_SIZE,
    "epochs": NUM_EPOCHS, "lr": LR,
    "num_classes": NUM_CLASSES, "class_names": class_names,
    "dataset_total": total_images,
    "train_samples": len(train_ds), "val_samples": len(val_ds),
    "test_samples": len(test_ds),
    "models": [{
        "name": r['label'], "timm_name": r['name'],
        "params_total": r['params_total'],
        "params_trainable": r['params_trainable'],
        "train_time_sec": round(r['time'], 1),
        "best_val_acc": round(r['best_val_acc'], 2),
        "test_acc": round(r['test_acc'], 2),
    } for r in results]
}
with open(os.path.join(OUTPUT_PATH, '07_summary.json'), 'w') as f:
    _json.dump(summary, f, indent=2, ensure_ascii=False)

total_time = sum(r['time'] for r in results)
print(f"\n{'='*60}")
print(f"🎉 완료! 총 학습: {total_time:.0f}s | Best: {best['label']} ({best['test_acc']:.2f}%)")
print(f"📁 {OUTPUT_PATH}")
print(f"{'='*60}")


✅ 로컬 데이터 이미 존재
⚡ Device: cuda
📂 Classes(10): ['Avulsion fracture', 'Comminuted fracture', 'Fracture Dislocation', 'Greenstick fracture', 'Hairline Fracture', 'Impacted fracture', 'Longitudinal fracture', 'Oblique fracture', 'Pathological fracture', 'Spiral Fracture']

📦 Train:903 | Val:112 | Test:114

🧠 EfficientNet-B0 — Partial Fine-Tuning
   Total: 4,020,358 | Trainable: 1,142,202 (28.4%)
  Epoch  1/10 | Train 2.222/19.4% | Val 2.173/20.5% ⭐
  Epoch  2/10 | Train 2.072/28.7% | Val 2.106/32.1% ⭐
  Epoch  3/10 | Train 1.938/34.2% | Val 2.023/27.7%
  Epoch  4/10 | Train 1.827/41.6% | Val 2.016/33.0% ⭐
  Epoch  5/10 | Train 1.764/40.9% | Val 1.978/31.2%
  Epoch  6/10 | Train 1.662/47.5% | Val 1.961/33.0%
  Epoch  7/10 | Train 1.595/49.3% | Val 1.953/32.1%
  Epoch  8/10 | Train 1.591/47.5% | Val 1.924/33.0%
  Epoch  9/10 | Train 1.581/49.3% | Val 1.949/31.2%
  Epoch 10/10 | Train 1.604/46.4% | Val 1.939/31.2%

  ✅ EfficientNet-B0 — Test Acc: 27.19% | Time: 75.0s

🧠 ResNet-50 — Partial Fin

Rank,Model,Total Params,Trainable,Train %,Time,Val Acc,Test Acc
🥇,ConvNeXt-Tiny,"27,835,508","15,487,508",55.6%,111s,50.0%,44.7%
🥈,DenseNet-121,"6,964,106","2,170,378",31.2%,87s,37.5%,32.5%
🥉,EfficientNet-B0,"4,020,358","1,142,202",28.4%,75s,33.0%,27.2%
4,MobileNetV2,"2,236,682","898,890",40.2%,73s,26.8%,26.3%
5,ResNet-50,"23,549,012","14,985,226",63.6%,91s,19.6%,15.8%







🏆 Best: ConvNeXt-Tiny — Classification Report
                       precision    recall  f1-score   support

    Avulsion fracture      0.500     0.273     0.353        11
  Comminuted fracture      0.583     0.467     0.519        15
 Fracture Dislocation      0.500     0.353     0.414        17
  Greenstick fracture      0.615     0.800     0.696        10
    Hairline Fracture      0.278     0.455     0.345        11
    Impacted fracture      0.455     0.625     0.526         8
Longitudinal fracture      1.000     0.083     0.154        12
     Oblique fracture      0.353     0.545     0.429        11
Pathological fracture      0.471     0.667     0.552        12
      Spiral Fracture      0.286     0.286     0.286         7

             accuracy                          0.447       114
            macro avg      0.504     0.455     0.427       114
         weighted avg      0.519     0.447     0.428       114


🎉 완료! 총 학습: 437s | Best: ConvNeXt-Tiny (44.74%)
📁 /content/driv